# OBLOKOS Phase I Migration Audit Notebook

"
"Assumptions logged for this audit run:
"
"- Local filesystem is the source of truth for pre-migration consistency checks.
"
"- Canonical identifier is the metadata filename stem (`<ERROR>`), and art should be `<ERROR>.png`.
"
"- Current tokenURI pattern is expected to resolve to `https://github.oblokos.com/metadata/<ERROR>.json`.
"
"- This notebook is read-only with respect to blockchain state: no transactions are made.
"
"- Optional on-chain checks (Section G) only run when RPC/contract env vars are provided.


In [ ]:
# %pip install pandas
# %pip install web3
# %pip install python-dotenv

## SECTION A - Setup

In [ ]:
import os
import json
import re
import hashlib
from pathlib import Path
from urllib.parse import urlparse, unquote

import pandas as pd
import numpy as np

RUN_TS = pd.Timestamp.utcnow().isoformat()
WORKDIR = Path.cwd()
METADATA_DIR = (WORKDIR / "metadata").resolve()
ART_DIR = (WORKDIR / "art").resolve()

REQUIRED_KEYS = ["id", "name", "description", "image", "external_url", "attributes"]
RECOMMENDED_KEYS = ["opensea_url"]
INT_EXPECTED_TRAIT_PATTERN = re.compile(r"(frame|level|index|rank|tier|step|phase|id)$", re.IGNORECASE)

print(f"Run timestamp (UTC): {RUN_TS}")
print(f"Working directory: {WORKDIR}")
print(f"METADATA_DIR: {METADATA_DIR} (exists={METADATA_DIR.exists()})")
print(f"ART_DIR: {ART_DIR} (exists={ART_DIR.exists()})")


def safe_read_json(path: Path):
    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        return data, None
    except Exception as e:
        return None, str(e)


def canonical_json_sha256(data):
    canonical = json.dumps(data, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


def extract_image_filename(image_field):
    if not isinstance(image_field, str) or not image_field.strip():
        return ""
    cleaned = image_field.strip()
    parsed = urlparse(cleaned)
    path_part = parsed.path if parsed.path else cleaned
    filename = Path(unquote(path_part)).name
    return filename


def merge_msgs(*parts):
    tokens = []
    for part in parts:
        if not part:
            continue
        if isinstance(part, str):
            chunk = [x.strip() for x in part.split(";") if x.strip()]
        else:
            chunk = [str(x).strip() for x in part if str(x).strip()]
        tokens.extend(chunk)
    dedup = sorted(set(tokens))
    return "; ".join(dedup)


## SECTION B - Load File Inventory

In [ ]:
metadata_files = sorted(METADATA_DIR.glob("*.json"))
art_files = sorted(ART_DIR.glob("*.png"))

metadata_inventory = pd.DataFrame(
    {
        "json_path": [str(p) for p in metadata_files],
        "error_name": [p.stem for p in metadata_files],
    }
)

art_inventory = pd.DataFrame(
    {
        "art_path": [str(p) for p in art_files],
        "error_name": [p.stem for p in art_files],
    }
)

metadata_names = set(metadata_inventory["error_name"]) if not metadata_inventory.empty else set()
art_names = set(art_inventory["error_name"]) if not art_inventory.empty else set()

missing_art = sorted(metadata_names - art_names)
orphan_art = sorted(art_names - metadata_names)

print(f"Metadata JSON files: {len(metadata_files)}")
print(f"Art PNG files: {len(art_files)}")
print(f"Missing art for metadata: {len(missing_art)}")
print(f"Orphan art without metadata: {len(orphan_art)}")

if missing_art:
    print("Sample missing art:", missing_art[:10])
if orphan_art:
    print("Sample orphan art:", orphan_art[:10])


## SECTION C - Validate Metadata Schema (T1.1)

In [ ]:
records = []

for json_path in metadata_files:
    error_name = json_path.stem
    data, parse_error = safe_read_json(json_path)

    errors = []
    warnings = []

    json_valid = parse_error is None
    schema_ok = False
    id_in_json = None
    name_in_json = None
    filename_matches_name = False
    image_field = ""
    image_filename_matches = False
    external_url_present = False
    attributes_ok = False
    json_sha256 = ""

    if not json_valid:
        errors.append(f"json_parse_error:{parse_error}")
    elif not isinstance(data, dict):
        errors.append("metadata_root_not_object")
    else:
        missing_required = [k for k in REQUIRED_KEYS if k not in data]
        if missing_required:
            errors.append(f"missing_required_keys:{','.join(missing_required)}")

        for k in RECOMMENDED_KEYS:
            if k not in data:
                warnings.append(f"missing_recommended_key:{k}")

        id_in_json = data.get("id")
        name_in_json = data.get("name")
        filename_matches_name = name_in_json == error_name
        if not filename_matches_name:
            errors.append("filename_mismatch_name")

        image_field = data.get("image", "")
        expected_image = f"{error_name}.png"
        extracted_image = extract_image_filename(image_field)
        image_filename_matches = extracted_image == expected_image
        if not image_filename_matches:
            errors.append(f"image_filename_mismatch:expected={expected_image},got={extracted_image or 'EMPTY'}")

        external_url = data.get("external_url", "")
        external_url_present = isinstance(external_url, str) and bool(external_url.strip())
        if not external_url_present:
            errors.append("external_url_missing_or_empty")

        attrs = data.get("attributes")
        attributes_ok = True
        if not isinstance(attrs, list):
            attributes_ok = False
            errors.append("attributes_not_list")
        else:
            for i, item in enumerate(attrs):
                if not isinstance(item, dict):
                    attributes_ok = False
                    errors.append(f"attributes[{i}]_not_object")
                    continue
                if "trait_type" not in item or "value" not in item:
                    attributes_ok = False
                    errors.append(f"attributes[{i}]_missing_trait_type_or_value")
                    continue

                trait_type = str(item.get("trait_type", ""))
                value = item.get("value")
                if INT_EXPECTED_TRAIT_PATTERN.search(trait_type) and isinstance(value, float):
                    warnings.append(f"float_where_int_expected:attributes.{trait_type}={value}")

        if isinstance(id_in_json, float):
            warnings.append(f"float_where_int_expected:id={id_in_json}")

        schema_ok = (
            len(missing_required) == 0
            and external_url_present
            and attributes_ok
        )

        try:
            json_sha256 = canonical_json_sha256(data)
        except Exception as e:
            errors.append(f"hash_error:{e}")

    records.append(
        {
            "error_name": error_name,
            "json_path": str(json_path),
            "json_valid": bool(json_valid),
            "schema_ok": bool(schema_ok),
            "id_in_json": id_in_json,
            "name_in_json": name_in_json,
            "filename_matches_name": bool(filename_matches_name),
            "image_field": image_field,
            "image_filename_matches": bool(image_filename_matches),
            "external_url_present": bool(external_url_present),
            "attributes_ok": bool(attributes_ok),
            "warnings": merge_msgs(warnings),
            "errors": merge_msgs(errors),
            "json_sha256": json_sha256,
        }
    )

metadata_df = pd.DataFrame(records)

if not metadata_df.empty:
    valid_id_mask = metadata_df["json_valid"] & metadata_df["id_in_json"].notna()
    duplicate_ids = (
        metadata_df.loc[valid_id_mask, "id_in_json"]
        .astype(str)
        .loc[lambda s: s.duplicated(keep=False)]
        .unique()
        .tolist()
    )

    if duplicate_ids:
        dup_mask = metadata_df["id_in_json"].astype(str).isin(duplicate_ids)
        metadata_df.loc[dup_mask, "errors"] = metadata_df.loc[dup_mask, "errors"].apply(
            lambda s: merge_msgs(s, "duplicate_id_in_json")
        )

metadata_df = metadata_df[
    [
        "error_name",
        "json_path",
        "json_valid",
        "schema_ok",
        "id_in_json",
        "name_in_json",
        "filename_matches_name",
        "image_field",
        "image_filename_matches",
        "external_url_present",
        "attributes_ok",
        "warnings",
        "errors",
        "json_sha256",
    ]
]

print("metadata_df rows:", len(metadata_df))
print("metadata parse failures:", int((~metadata_df["json_valid"]).sum()) if not metadata_df.empty else 0)
print("metadata schema failures:", int((~metadata_df["schema_ok"]).sum()) if not metadata_df.empty else 0)


## SECTION D - Validate Art Files (T1.1)

In [ ]:
art_records = []

for png_path in art_files:
    exists = png_path.exists()
    size_bytes = png_path.stat().st_size if exists else 0
    art_records.append(
        {
            "error_name": png_path.stem,
            "art_path": str(png_path),
            "art_exists": bool(exists),
            "art_size_bytes": int(size_bytes),
        }
    )

art_df = pd.DataFrame(art_records, columns=["error_name", "art_path", "art_exists", "art_size_bytes"])

print("art_df rows:", len(art_df))
if not art_df.empty:
    print("empty/zero-byte PNG files:", int(((art_df["art_exists"]) & (art_df["art_size_bytes"] <= 0)).sum()))


## SECTION E - Merge and Issue Report

In [ ]:
audit_df = pd.merge(metadata_df, art_df, on="error_name", how="outer", indicator=True)

bool_cols = [
    "json_valid",
    "schema_ok",
    "filename_matches_name",
    "image_filename_matches",
    "external_url_present",
    "attributes_ok",
    "art_exists",
]
for col in bool_cols:
    if col not in audit_df:
        audit_df[col] = False
    audit_df[col] = audit_df[col].fillna(False).astype(bool)

if "art_size_bytes" not in audit_df:
    audit_df["art_size_bytes"] = 0
audit_df["art_size_bytes"] = pd.to_numeric(audit_df["art_size_bytes"], errors="coerce").fillna(0).astype("int64")

for col in ["warnings", "errors", "json_path", "art_path", "name_in_json", "image_field", "id_in_json", "json_sha256"]:
    if col not in audit_df:
        audit_df[col] = ""
    audit_df[col] = audit_df[col].fillna("")


def compute_row_error(row):
    extra = []
    if row["_merge"] == "left_only":
        extra.append("missing_art_file")
    if row["_merge"] == "right_only":
        extra.append("orphan_art_without_metadata")
    if row["art_exists"] and int(row["art_size_bytes"]) <= 0:
        extra.append("art_zero_bytes")
    if not row["art_exists"]:
        extra.append("art_missing_or_unreadable")
    return merge_msgs(row.get("errors", ""), extra)


audit_df["errors"] = audit_df.apply(compute_row_error, axis=1)

critical_cols = [
    "json_valid",
    "schema_ok",
    "filename_matches_name",
    "image_filename_matches",
    "external_url_present",
    "attributes_ok",
    "art_exists",
]

audit_df["ok"] = audit_df[critical_cols].all(axis=1) & (audit_df["errors"].str.strip() == "")
issues_df = audit_df[(~audit_df["ok"]) | (audit_df["errors"].str.strip() != "")].copy()

audit_df = audit_df.sort_values("error_name").reset_index(drop=True)
issues_df = issues_df.sort_values("error_name").reset_index(drop=True)

audit_df.to_csv("audit_metadata.csv", index=False)
issues_df.to_csv("audit_issues.csv", index=False)

metadata_total_bytes = int(sum(p.stat().st_size for p in metadata_files))
art_total_bytes = int(sum(p.stat().st_size for p in art_files))
combined_total_bytes = metadata_total_bytes + art_total_bytes

error_tokens = []
for s in issues_df["errors"].tolist():
    error_tokens.extend([t.strip() for t in str(s).split(";") if t.strip()])

top_issues = (
    pd.Series(error_tokens).value_counts().head(20).to_dict() if error_tokens else {}
)

summary = {
    "run_timestamp_utc": RUN_TS,
    "paths": {
        "workdir": str(WORKDIR),
        "metadata_dir": str(METADATA_DIR),
        "art_dir": str(ART_DIR),
    },
    "counts": {
        "metadata_files": int(len(metadata_files)),
        "art_files": int(len(art_files)),
        "audit_rows": int(len(audit_df)),
        "ok_rows": int(audit_df["ok"].sum()) if not audit_df.empty else 0,
        "issue_rows": int(len(issues_df)),
        "missing_art_for_metadata": int(len(missing_art)),
        "orphan_art_without_metadata": int(len(orphan_art)),
    },
    "sizes_bytes": {
        "metadata_total": metadata_total_bytes,
        "art_total": art_total_bytes,
        "combined_total": combined_total_bytes,
    },
    "top_issues": top_issues,
}

Path("audit_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Wrote: audit_metadata.csv, audit_issues.csv, audit_summary.json")
print("Total rows:", len(audit_df), "| Issues:", len(issues_df), "| OK:", int(audit_df["ok"].sum()))
if top_issues:
    print("Top issues (up to 20):")
    for k, v in top_issues.items():
        print(f"  - {k}: {v}")


## SECTION E.1 - Teleport Lab Cross-check

Assumptions for this section:
- `teleport_lab_assets_nft.csv` is the curated list of NFTs used in Teleport Lab.
- The `filename` column contains canonical error names and should match `error_name` in the audit.
- This check is local and read-only; it does not query chain data.


In [ ]:
TELEPORT_CSV_PATH = (WORKDIR / "teleport_lab_assets_nft.csv").resolve()
TELEPORT_ISSUES_OUT = WORKDIR / "teleport_lab_issues_overlap.csv"

if not TELEPORT_CSV_PATH.exists():
    print(f"Teleport list not found: {TELEPORT_CSV_PATH}")
    teleport_df = pd.DataFrame(columns=["tokenID", "filename", "externalLink", "description", "error_name"])
else:
    teleport_df = pd.read_csv(TELEPORT_CSV_PATH, dtype=str, keep_default_na=False)

required_cols = ["tokenID", "filename"]
missing_cols = [c for c in required_cols if c not in teleport_df.columns]
if missing_cols:
    raise ValueError(f"teleport_lab_assets_nft.csv missing required columns: {missing_cols}")

teleport_df["error_name"] = (
    teleport_df["filename"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.(json|png)$", "", regex=True, case=False)
)
teleport_df = teleport_df[teleport_df["error_name"] != ""].copy()

audit_subset_cols = ["error_name", "ok", "errors", "warnings"]
audit_subset = audit_df[audit_subset_cols].copy()

teleport_audit_df = teleport_df.merge(audit_subset, on="error_name", how="left")
teleport_audit_df["in_local_audit"] = teleport_audit_df["ok"].notna()
teleport_audit_df["has_issue"] = (~teleport_audit_df["ok"].fillna(False))
teleport_audit_df.loc[~teleport_audit_df["in_local_audit"], "errors"] = teleport_audit_df.loc[~teleport_audit_df["in_local_audit"], "errors"].fillna("missing_from_local_audit_inventory")
teleport_audit_df["errors"] = teleport_audit_df["errors"].fillna("")
teleport_audit_df["warnings"] = teleport_audit_df["warnings"].fillna("")

teleport_issues_df = teleport_audit_df[teleport_audit_df["has_issue"]].copy()
teleport_issues_df = teleport_issues_df.sort_values(["error_name", "tokenID"]).reset_index(drop=True)
teleport_issues_df.to_csv(TELEPORT_ISSUES_OUT, index=False)

print(f"Teleport Lab rows: {len(teleport_df)}")
print(f"Teleport Lab present in local audit: {int(teleport_audit_df['in_local_audit'].sum())}")
print(f"Teleport Lab with issues: {len(teleport_issues_df)}")
print(f"Wrote: {TELEPORT_ISSUES_OUT}")

if not teleport_issues_df.empty:
    print("Top Teleport Lab issues (up to 20 rows):")
    print(teleport_issues_df[["tokenID", "error_name", "errors"]].head(20).to_string(index=False))
else:
    print("No Teleport Lab NFTs appear in the current issues list.")


## SECTION F - Size Analysis (T1.3)

In [ ]:
size_rows = []

for p in metadata_files:
    size_rows.append(
        {
            "file_type": "metadata_json",
            "error_name": p.stem,
            "path": str(p),
            "size_bytes": int(p.stat().st_size),
        }
    )

for p in art_files:
    size_rows.append(
        {
            "file_type": "art_png",
            "error_name": p.stem,
            "path": str(p),
            "size_bytes": int(p.stat().st_size),
        }
    )

size_df = pd.DataFrame(size_rows)
size_df = size_df.sort_values(["file_type", "size_bytes"], ascending=[True, False]).reset_index(drop=True)
size_df.to_csv("size_report.csv", index=False)

image_sizes = size_df.loc[size_df["file_type"] == "art_png", "size_bytes"]
avg_image_size = float(image_sizes.mean()) if not image_sizes.empty else 0.0
median_image_size = float(image_sizes.median()) if not image_sizes.empty else 0.0
largest_20_images = (
    size_df[size_df["file_type"] == "art_png"]
    .nlargest(20, "size_bytes")
    [["error_name", "size_bytes", "path"]]
)

print("Wrote: size_report.csv")
print(f"Metadata bytes total: {metadata_total_bytes:,}")
print(f"Art bytes total: {art_total_bytes:,}")
print(f"Combined bytes total: {combined_total_bytes:,}")
print(f"Average image size (bytes): {avg_image_size:,.2f}")
print(f"Median image size (bytes): {median_image_size:,.2f}")
print("Largest 20 images:")
print(largest_20_images.to_string(index=False))


### Arweave Cost Estimation Notes

"
"Use the output `combined_total_bytes` from Section F and set your market price input:

"
"- `total_gb = combined_total_bytes / (1024 ** 3)`
"
"- `estimated_cost = total_gb * price_per_gb`

"
"Keep `price_per_gb` as a user input value so pricing can be updated at execution time.


In [ ]:
# User input cell: set the current market rate for Arweave storage
price_per_gb = np.nan  # Example: 7.5

total_gb = combined_total_bytes / (1024 ** 3)
if pd.notna(price_per_gb):
    estimated_cost = total_gb * float(price_per_gb)
    print(f"total_gb: {total_gb:.6f}")
    print(f"estimated_cost: {estimated_cost:.6f}")
else:
    print(f"total_gb: {total_gb:.6f}")
    print("Set price_per_gb to compute estimated_cost.")


## SECTION G (Optional) - On-chain Consistency (T1.2)

In [ ]:
import importlib.util

WEB3_AVAILABLE = importlib.util.find_spec("web3") is not None
print(f"web3 available: {WEB3_AVAILABLE}")
if not WEB3_AVAILABLE:
    print("Install dependency if needed: python -m pip install web3")

DOTENV_AVAILABLE = importlib.util.find_spec("dotenv") is not None
print(f"python-dotenv available: {DOTENV_AVAILABLE}")
if not DOTENV_AVAILABLE:
    print("Install dependency if needed: python -m pip install python-dotenv")
else:
    from dotenv import load_dotenv
    import os

    load_dotenv()  # carga .env automáticamente

WEB3_AVAILABLE = False  # Set to True if you have web3 installed and want to enable blockchain checks


If `web3` is available, configure env vars before running the next cell:

"
"- `POLYGON_RPC_URL` (HTTP RPC endpoint)
"
"- `OBLOKOS_CONTRACT_ADDRESS` (ERC-721 enumerable contract)

"
"Security note: keep RPC URL in env vars; do not hardcode secrets and do not print private credentials.


In [ ]:
chain_audit_columns = [
    "token_id",
    "token_uri",
    "error_name",
    "local_metadata_exists",
    "status",
]

chain_audit_df = pd.DataFrame(columns=chain_audit_columns)

if not WEB3_AVAILABLE:
    print("Skipping T1.2 on-chain audit: web3 not installed.")
else:
    from web3 import Web3

    polygon_rpc_url = os.getenv("POLYGON_RPC_URL", "").strip()
    contract_address = os.getenv("OBLOKOS_CONTRACT_ADDRESS", "").strip()

    if not polygon_rpc_url or not contract_address:
        print("Skipping T1.2 on-chain audit: set POLYGON_RPC_URL and OBLOKOS_CONTRACT_ADDRESS env vars.")
    else:
        abi = [
            {
                "inputs": [],
                "name": "totalSupply",
                "outputs": [{"internalType": "uint256", "name": "", "type": "uint256"}],
                "stateMutability": "view",
                "type": "function",
            },
            {
                "inputs": [{"internalType": "uint256", "name": "index", "type": "uint256"}],
                "name": "tokenByIndex",
                "outputs": [{"internalType": "uint256", "name": "", "type": "uint256"}],
                "stateMutability": "view",
                "type": "function",
            },
            {
                "inputs": [{"internalType": "uint256", "name": "tokenId", "type": "uint256"}],
                "name": "tokenURI",
                "outputs": [{"internalType": "string", "name": "", "type": "string"}],
                "stateMutability": "view",
                "type": "function",
            },
        ]

        w3 = Web3(Web3.HTTPProvider(polygon_rpc_url, request_kwargs={"timeout": 20}))
        if not w3.is_connected():
            print("Skipping T1.2 on-chain audit: unable to connect to Polygon RPC.")
        else:
            contract = w3.eth.contract(address=Web3.to_checksum_address(contract_address), abi=abi)
            rows = []
            try:
                total_supply = int(contract.functions.totalSupply().call())
                print(f"On-chain totalSupply: {total_supply}")

                for i in range(total_supply):
                    token_id = int(contract.functions.tokenByIndex(i).call())
                    token_uri = str(contract.functions.tokenURI(token_id).call())
                    token_file = Path(unquote(urlparse(token_uri).path)).name
                    token_error = token_file[:-5] if token_file.lower().endswith(".json") else token_file

                    rows.append(
                        {
                            "token_id": token_id,
                            "token_uri": token_uri,
                            "error_name": token_error,
                            "local_metadata_exists": token_error in metadata_names,
                            "status": "ok" if token_error in metadata_names else "token_present_file_missing",
                        }
                    )

                chain_audit_df = pd.DataFrame(rows, columns=chain_audit_columns)

                chain_names = set(chain_audit_df["error_name"].dropna().astype(str).tolist())
                file_only = sorted(metadata_names - chain_names)

                if file_only:
                    extra = pd.DataFrame(
                        [
                            {
                                "token_id": np.nan,
                                "token_uri": "",
                                "error_name": n,
                                "local_metadata_exists": True,
                                "status": "file_present_token_missing",
                            }
                            for n in file_only
                        ],
                        columns=chain_audit_columns,
                    )
                    chain_audit_df = pd.concat([chain_audit_df, extra], ignore_index=True)

            except Exception as e:
                print(f"T1.2 on-chain audit failed: {e}")

if chain_audit_df.empty:
    chain_audit_df = pd.DataFrame(
        [
            {
                "token_id": np.nan,
                "token_uri": "",
                "error_name": "",
                "local_metadata_exists": np.nan,
                "status": "not_run",
            }
        ],
        columns=chain_audit_columns,
    )

chain_audit_df.to_csv("chain_audit.csv", index=False)
print("Wrote: chain_audit.csv")
print(chain_audit_df.head(10).to_string(index=False))
